# 00 - Cluster Spark

Aquí revisamos que el notebook sí esté conectado al cluster: el master, los executors, cómo se reparten las tareas y que HDFS funcione.

## 1. Sesión de Spark

No hace falta poner el master ni la memoria porque ya están en `spark-defaults.conf`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("00_cluster")
    .getOrCreate()
)
sc = spark.sparkContext

print("Versión de Spark :", spark.version)
print("Master           :", sc.master)
print("Id de aplicación :", sc.applicationId)
print("UI de la app     :", sc.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/27 16:54:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versión de Spark : 4.1.3
Master           : spark://10.43.97.45:7077
Id de aplicación : app-20260927165447-0010
UI de la app     : http://10.43.97.45:4040


Si `Master` empieza por `spark://` estamos en el cluster. Si sale `local[*]`, algo quedó mal y todo está corriendo solo en esta máquina.

## 2. Configuración

In [2]:
import pandas as pd

claves = [
    "spark.master", "spark.driver.host", "spark.driver.memory",
    "spark.executor.cores", "spark.executor.memory",
    "spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled",
    "spark.serializer", "spark.sql.session.timeZone",
]
conf = dict(sc.getConf().getAll())
pd.DataFrame({"parámetro": claves, "valor": [conf.get(k, "(por defecto)") for k in claves]})

,parámetro,valor
0,spark.master,spark://10.43.97.45:7077
1,spark.driver.host,10.43.97.45
2,spark.driver.memory,3g
3,spark.executor.cores,1
4,spark.executor.memory,2560m
5,spark.sql.shuffle.partitions,24
6,spark.sql.adaptive.enabled,true
7,spark.serializer,org.apache.spark.serializer.KryoSerializer
8,spark.sql.session.timeZone,America/New_York


## 3. Executors

In [3]:
import requests

url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"
ejecutores = pd.DataFrame(requests.get(url, timeout=10).json())
ejecutores["memoria_GB"] = (ejecutores["maxMemory"] / 1024**3).round(2)
tabla = ejecutores[["id", "hostPort", "totalCores", "memoria_GB", "isActive"]]
tabla

,id,hostPort,totalCores,memoria_GB,isActive
0,driver,10.43.97.45:40001,0,1.62,True
1,7,10.43.97.45:40011,1,1.32,True
2,6,10.43.97.45:40010,1,1.32,True
3,5,10.43.97.50:40011,1,1.32,True
4,4,10.43.97.50:40012,1,1.32,True
5,3,10.43.97.50:40010,1,1.32,True
6,2,10.43.97.54:40011,1,1.32,True
7,1,10.43.97.54:40012,1,1.32,True
8,0,10.43.97.54:40010,1,1.32,True


In [4]:
por_maquina = (
    tabla[tabla["id"] != "driver"]
    .assign(maquina=lambda d: d["hostPort"].str.split(":").str[0])
    .groupby("maquina")
    .agg(executors=("id", "count"), cores=("totalCores", "sum"), memoria_GB=("memoria_GB", "sum"))
)
print(f"Paralelismo por defecto: {sc.defaultParallelism} tareas simultáneas")
por_maquina

Paralelismo por defecto: 8 tareas simultáneas


,executors,cores,memoria_GB
maquina,,,
10.43.97.45,2,2,2.64
10.43.97.50,3,3,3.96
10.43.97.54,3,3,3.96


Tienen que aparecer las tres VMs.

## 4. Reparto de tareas

Lanzamos 200 tareas y cada una devuelve el nombre de la máquina donde corrió.

In [5]:
import os, socket

def donde_corro(_):
    return f"{socket.gethostname()} ({os.environ.get('SPARK_LOCAL_IP', '?')})"

reparto = sc.parallelize(range(200), 200).map(donde_corro).countByValue()
pd.Series(dict(reparto), name="tareas").to_frame()

,tareas
NBDG37 (10.43.97.54),78
NBDG28 (10.43.97.45),40
NBDG33 (10.43.97.50),82


## 5. HDFS

Prueba de escritura y lectura: 10 millones de filas a Parquet y de vuelta.

In [6]:
import time
from pyspark.sql import functions as F

ruta = "/tmp/prueba_00_cluster.parquet"   # ruta en HDFS

t0 = time.time()
(spark.range(10_000_000)
      .withColumn("grupo", F.col("id") % 5)
      .withColumn("valor", F.rand(seed=42))
      .write.mode("overwrite").parquet(ruta))
t_escritura = time.time() - t0

t0 = time.time()
resumen = spark.read.parquet(ruta).groupBy("grupo").agg(F.count("*").alias("filas"), F.avg("valor").alias("promedio"))
resultado = resumen.orderBy("grupo").toPandas()
t_lectura = time.time() - t0

print(f"Escritura: {t_escritura:.1f} s  |  Lectura + agregación: {t_lectura:.1f} s")
resultado

Escritura: 10.1 s  |  Lectura + agregación: 6.2 s


,grupo,filas,promedio
0,0,2000000,0.499898
1,1,2000000,0.499586
2,2,2000000,0.499983
3,3,2000000,0.499897
4,4,2000000,0.499998


In [7]:
# borrar la prueba
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
print("Sistema de archivos por defecto:", fs.getUri())
fs.delete(jvm.org.apache.hadoop.fs.Path(ruta), True)

Sistema de archivos por defecto: hdfs://10.43.97.45:9000


True

Cada grupo debería tener 2.000.000 de filas y un promedio cerca de 0,5, y el sistema de archivos tiene que ser `hdfs://`.

## Conclusiones

- El cluster es Spark standalone: VM1 tiene el master, un worker y el driver, y VM2 y VM3 tienen un worker cada una. HDFS está con replicación 2.
- La sesión quedó conectada al master, hay executors en las tres VMs, las tareas se reparten entre ellas y HDFS lee y escribe sin problema.